# 7 - Sentiment Analysis IndoBERT

## 1. Install dan setup

In [1]:
!pip install -q pandas numpy transformers torch accelerate openpyxl tqdm

from pathlib import Path
import os, re, shutil, json, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

BASE_DIR = Path("/content")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
EDA_DIR = OUTPUT_DIR / "eda"
SENTIMENT_DIR = OUTPUT_DIR / "sentiment"
MODEL_DIR = OUTPUT_DIR / "models"
FINAL_DIR = OUTPUT_DIR / "final"

for d in [DATA_DIR, OUTPUT_DIR, EDA_DIR, SENTIMENT_DIR, MODEL_DIR, FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def upload_file_colab(prompt="Upload file CSV"):
    """Upload file di Google Colab dan return Path file pertama."""
    from google.colab import files
    print(prompt)
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("Tidak ada file yang di-upload.")
    filename = list(uploaded.keys())[0]
    return Path(filename)


def find_or_upload(candidates, upload_prompt):
    """
    Cari file dari daftar kandidat. Kalau tidak ada, minta upload.
    Cocok untuk Colab: bisa lanjut dari file hasil notebook sebelumnya.
    """
    for p in candidates:
        p = Path(p)
        if p.exists():
            print(f"File ditemukan: {p}")
            return p
    return upload_file_colab(upload_prompt)


def normalize_sentiment_label(label):
    """Samakan label menjadi Positive / Neutral / Negative."""
    if pd.isna(label):
        return np.nan
    text = str(label).strip().lower()

    positive_values = {"positive", "positif", "pos", "1", "label_0"}
    neutral_values = {"neutral", "netral", "neu", "0", "label_1"}
    negative_values = {"negative", "negatif", "neg", "-1", "label_2"}

    if text in positive_values:
        return "Positive"
    if text in neutral_values:
        return "Neutral"
    if text in negative_values:
        return "Negative"
    return np.nan


def standardize_dataframe(df):
    """
    Standarisasi kolom dari hasil preprocessing 2a.
    File sumber biasanya punya: Link, Judul, Isi Berita, Status, Tag, Sentimen, Penerbit,
    final_text_clean, final_text_stemmed, jumlah_token_before_sw, jumlah_token_after_sw.
    """
    df = df.copy()

    rename_map = {
        "Link": "source_url",
        "Judul": "title",
        "Isi Berita": "content",
        "Status": "scrape_status",
        "Tag": "tag",
        "Sentimen": "sentiment_original",
        "Sentiment": "sentiment_original",
        "Penerbit": "publisher",
        "Tanggal": "published_date"
    }

    for old, new in rename_map.items():
        if old in df.columns and new not in df.columns:
            df[new] = df[old]

    # Label asli/manual untuk evaluasi
    if "label_true" not in df.columns:
        if "sentiment_original" in df.columns:
            df["label_true"] = df["sentiment_original"].apply(normalize_sentiment_label)
        else:
            df["label_true"] = np.nan

    # Kolom teks fallback
    if "final_text_clean" not in df.columns:
        if "content" in df.columns:
            df["final_text_clean"] = df["content"].fillna("").astype(str).str.lower()
        else:
            raise KeyError("Tidak ditemukan kolom final_text_clean atau content/Isi Berita.")

    if "final_text_stemmed" not in df.columns:
        df["final_text_stemmed"] = df["final_text_clean"]

    # Token count fallback
    if "jumlah_token_before_sw" not in df.columns:
        df["jumlah_token_before_sw"] = df["final_text_clean"].fillna("").astype(str).str.split().apply(len)
    if "jumlah_token_after_sw" not in df.columns:
        df["jumlah_token_after_sw"] = df["final_text_stemmed"].fillna("").astype(str).str.split().apply(len)

    # Filter success kalau kolom status ada
    if "scrape_status" in df.columns:
        success_mask = df["scrape_status"].astype(str).str.lower().eq("success")
        if success_mask.sum() > 0:
            df = df[success_mask].copy()

    # Buang teks kosong
    df["final_text_clean"] = df["final_text_clean"].fillna("").astype(str)
    df["final_text_stemmed"] = df["final_text_stemmed"].fillna("").astype(str)
    df = df[df["final_text_clean"].str.strip().ne("")].copy()
    df = df.reset_index(drop=True)

    return df


def load_after_2a_csv(path):
    """Load CSV hasil preprocessing 2a dengan fallback encoding."""
    path = Path(path)
    try:
        df = pd.read_csv(path)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin-1")
    return standardize_dataframe(df)


def safe_display(obj, n=5):
    try:
        display(obj)
    except Exception:
        print(obj.head(n) if hasattr(obj, "head") else obj)


## 2. Input data
Gunakan hasil Logistic Regression jika ada, atau upload file CSV hasil notebook sebelumnya.

In [2]:

INPUT_PATH = find_or_upload(
    candidates=[
        SENTIMENT_DIR / "hasil_textblob_logreg_sentiment.csv",
        "/content/hasil_textblob_logreg_sentiment.csv",
        SENTIMENT_DIR / "hasil_textblob_sentiment.csv",
        "/content/hasil_textblob_sentiment.csv",
        DATA_DIR / "hasil_preprocessing_pertamina_standardized.csv",
        "/content/hasil_preprocessing_pertamina.csv",
    ],
    upload_prompt="Upload hasil_textblob_logreg_sentiment.csv atau file hasil preprocessing"
)

df = load_after_2a_csv(INPUT_PATH)
print("Input:", INPUT_PATH)
print("Shape:", df.shape)
safe_display(df.head(3))


Upload hasil_textblob_logreg_sentiment.csv atau file hasil preprocessing


Saving hasil_textblob_logreg_sentiment.csv to hasil_textblob_logreg_sentiment.csv
Input: hasil_textblob_logreg_sentiment.csv
Shape: (1829, 28)


,Link,Judul,Isi Berita,Status,Tag,Sentimen,Penerbit,final_text_clean,final_text_stemmed,jumlah_token_before_sw,...,label_true,textblob_polarity,textblob_subjectivity,textblob_sentiment,logreg_sentiment,split,logreg_prob_Negative,logreg_prob_Neutral,logreg_prob_Positive,logreg_confidence
0,https://money.kompas.com/image/2017/10/03/1815...,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,Diskusi mengenai CSR di industri pelumas berta...,success,Ekonomi,Neutral,Kompas,diskusi mengenai csr industri pelumas bertajuk...,diskusi kena csr industri lumas tajuk cari for...,16,...,Neutral,0.0,0.0,Neutral,Neutral,train,0.166593,0.518138,0.315269,0.518138
1,https://money.kompas.com/read/2016/06/06/14280...,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...","JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...",success,Kecelakaan Kerja,Positive,Kompas,jakarta kompas com anak perusahaan pertamina y...,anak usaha pertamina pertamina lubricants gand...,130,...,Positive,0.0,0.0,Neutral,Positive,test,0.116579,0.331690,0.551732,0.551732
2,https://money.kompas.com/read/2017/02/03/12195...,"Dua Pucuk Pimpinan Pertamina Dicopot, Yenni An...","JAKARTA, KOMPAS.com — Pasca-pencopotan dua puc...",success,Migas,Positive,Kompas,jakarta kompas com pasca pencopotan dua pucuk ...,pasca copot pucuk pimpin pertamina menteri bad...,145,...,Positive,0.0,0.0,Neutral,Neutral,train,0.136259,0.479430,0.384311,0.479430


## 3. IndoBERT sentiment

In [3]:

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm.auto import tqdm

MODEL_NAME = "mdhugol/indonesia-bert-sentiment-classification"
TEXT_COL_INDOBERT = "final_text_clean"

print("Model:", MODEL_NAME)
print("Kolom teks:", TEXT_COL_INDOBERT)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

print("id2label dari model:", model.config.id2label)

device = 0 if torch.cuda.is_available() else -1
print("Device:", "GPU" if device == 0 else "CPU")

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=device
)

INDOBERT_LABEL_MAP = {
    "LABEL_0": "Positive",
    "LABEL_1": "Neutral",
    "LABEL_2": "Negative",
    "positive": "Positive",
    "positif": "Positive",
    "neutral": "Neutral",
    "netral": "Neutral",
    "negative": "Negative",
    "negatif": "Negative",
    "Positive": "Positive",
    "Neutral": "Neutral",
    "Negative": "Negative",
}

def normalize_indobert_label(label):
    label = str(label).strip()
    return INDOBERT_LABEL_MAP.get(label, label)

texts = df[TEXT_COL_INDOBERT].fillna("").astype(str).tolist()
BATCH_SIZE = 16
results = []

for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i+BATCH_SIZE]
    out = sentiment_pipeline(batch, truncation=True, max_length=512)
    results.extend(out)

df["indobert_raw_label"] = [r["label"] for r in results]
df["indobert_score"] = [r["score"] for r in results]
df["indobert_sentiment"] = df["indobert_raw_label"].apply(normalize_indobert_label)

summary = df["indobert_sentiment"].value_counts().reindex(["Positive", "Neutral", "Negative"], fill_value=0).reset_index()
summary.columns = ["sentiment", "jumlah"]
safe_display(summary)
safe_display(df[[TEXT_COL_INDOBERT, "indobert_raw_label", "indobert_sentiment", "indobert_score"]].head(10))


Model: mdhugol/indonesia-bert-sentiment-classification
Kolom teks: final_text_clean


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: mdhugol/indonesia-bert-sentiment-classification
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


id2label dari model: {0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2'}
Device: CPU


  0%|          | 0/115 [00:00<?, ?it/s]

,sentiment,jumlah
0,Positive,515
1,Neutral,1002
2,Negative,312


,final_text_clean,indobert_raw_label,indobert_sentiment,indobert_score
0,diskusi mengenai csr industri pelumas bertajuk...,LABEL_1,Neutral,0.995060
1,jakarta kompas com anak perusahaan pertamina y...,LABEL_0,Positive,0.806849
2,jakarta kompas com pasca pencopotan dua pucuk ...,LABEL_1,Neutral,0.795107
3,jakarta kompas com pertamina persero telah mem...,LABEL_0,Positive,0.586893
4,jakarta kompas com adiatma sardjito telah resm...,LABEL_0,Positive,0.464756
5,jakarta kompas com pertamina persero akhirnya ...,LABEL_1,Neutral,0.563465
6,jakarta kompas com penyidik jaksa agung muda p...,LABEL_1,Neutral,0.749409
7,jakarta kompas com seorang warganet bernama pa...,LABEL_2,Negative,0.706556
8,jakarta kompasotomotif pertamina persero membe...,LABEL_1,Neutral,0.421760
9,pelepasan indukan tuntong laut oleh tim konser...,LABEL_1,Neutral,0.991437


## 4. Export hasil IndoBERT

In [4]:

OUTPUT_CSV = SENTIMENT_DIR / "hasil_all_methods_sentiment.csv"
OUTPUT_EXCEL = SENTIMENT_DIR / "hasil_all_methods_sentiment.xlsx"

df.to_csv(OUTPUT_CSV, index=False)
with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="All_Methods", index=False)
    summary.to_excel(writer, sheet_name="IndoBERT_Distribution", index=False)

print("CSV tersimpan:", OUTPUT_CSV)
print("Excel tersimpan:", OUTPUT_EXCEL)

from google.colab import files
files.download(str(OUTPUT_CSV))


CSV tersimpan: /content/outputs/sentiment/hasil_all_methods_sentiment.csv
Excel tersimpan: /content/outputs/sentiment/hasil_all_methods_sentiment.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>